### Model Experiments

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
from src.visualization import silhouette_plot
from src.topic_analysis import *
import pickle
import mlflow
import mlflow.sklearn

In [0]:
df = spark.table("scientific_trend_analysis.core.arxiv_cleaned_sampled")
df_label = df.toPandas()

In [0]:
processed_text_list = [row['processed_text'] for row in df.select('processed_text').collect()]
print(f"Loaded {len(processed_text_list)} processed texts")

#### Hyperparameter Tuning

In [0]:
%skip
# Set experiment name
current_user = spark.sql("SELECT current_user() as user").collect()[0]['user']
mlflow.set_experiment(f"/Users/{current_user}/scientific-trend-analysis-hyperparameter-tuning")

# Define hyperparameter grid
max_features_options = [20000, 30000]
threshold_options = [0.00001, 0.00005, 0.0001]
n_components_options = [50, 100, 200]

print(f"Total combinations: {len(max_features_options) * len(threshold_options) * len(n_components_options)}")
print("="*100)

experiment_results = []

for max_feat in max_features_options:
    for threshold in threshold_options:
        for n_comp in n_components_options:
            
            run_name = f"maxfeat_{max_feat}_thresh_{threshold}_ncomp_{n_comp}"
            print(f"\nRunning: {run_name}")
            
            with mlflow.start_run(run_name=run_name):
                # Log parameters
                mlflow.log_param("max_features", max_feat)
                mlflow.log_param("variance_threshold", threshold)
                mlflow.log_param("n_components", n_comp)
                mlflow.log_param("tfidf_max_df", 0.85)
                mlflow.log_param("tfidf_min_df", 5)
                mlflow.log_param("ngram_range", "(1, 2)")
                
                try:
                    # Step 1: TF-IDF Vectorization
                    tfidf_vec = TfidfVectorizer(
                        max_df=0.85, 
                        min_df=5, 
                        max_features=max_feat, 
                        sublinear_tf=True, 
                        stop_words='english', 
                        ngram_range=(1, 2)
                    )
                    X_tfidf_exp = tfidf_vec.fit_transform(processed_text_list)
                    
                    mlflow.log_metric("features_after_tfidf", X_tfidf_exp.shape[1])
                    
                    # Step 2: Variance Threshold
                    selector_exp = VarianceThreshold(threshold=threshold)
                    X_tfidf_reduced_exp = selector_exp.fit_transform(X_tfidf_exp)
                    
                    mlflow.log_metric("features_after_variance", X_tfidf_reduced_exp.shape[1])
                    
                    # Step 3: LSA (TruncatedSVD)
                    svd_exp = TruncatedSVD(n_components=n_comp, random_state=2)
                    lsa_result_exp = svd_exp.fit_transform(X_tfidf_reduced_exp)
                    
                    explained_var = svd_exp.explained_variance_ratio_.sum()
                    mlflow.log_metric("explained_variance_ratio", explained_var)
                    
                    # Store results
                    result = {
                        'max_features': max_feat,
                        'threshold': threshold,
                        'n_components': n_comp,
                        'features_after_tfidf': X_tfidf_exp.shape[1],
                        'features_after_variance': X_tfidf_reduced_exp.shape[1],
                        'explained_variance': explained_var
                    }
                    experiment_results.append(result)
                    
                    print(f"  TF-IDF Features: {X_tfidf_exp.shape[1]} → After Variance: {X_tfidf_reduced_exp.shape[1]}")
                    print(f"  Explained Variance: {explained_var:.4f} ({explained_var*100:.2f}%)")
                    
                except Exception as e:
                    print(f"  ERROR: {str(e)}")
                    mlflow.log_param("status", "failed")
                    mlflow.log_param("error", str(e))

In [0]:
%skip
# Convert results to DataFrame for analysis
results_df = pd.DataFrame(experiment_results)

# Explained Variance by N Components (grouped by max_features)
plt.figure(figsize=(10, 6))
for max_feat in sorted(results_df['max_features'].unique()):
    subset = results_df[results_df['max_features'] == max_feat].groupby('n_components')['explained_variance'].mean()
    plt.plot(subset.index, subset.values, marker='o', label=f'max_feat={max_feat}', linewidth=2, markersize=8)

plt.xlabel('N Components (LSA)', fontsize=12)
plt.ylabel('Explained Variance Ratio', fontsize=12)
plt.title('Explained Variance by N Components', fontsize=13, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.ylim([0, 1])

plt.tight_layout()
plt.show()

display(results_df.sort_values('explained_variance', ascending=False))

#### Dimensionality Reduction

In [0]:
tfidf_vectorizer = TfidfVectorizer(max_df=0.85, min_df=5, max_features=30000, sublinear_tf=True, stop_words='english', ngram_range=(1, 2))
X_tfidf = tfidf_vectorizer.fit_transform(processed_text_list)
feature_name = tfidf_vectorizer.get_feature_names_out()
print(feature_name)

selector = VarianceThreshold(threshold=0.0001)
X_tfidf_reduced = selector.fit_transform(X_tfidf)
feature_name_reduced = feature_name[selector.get_support()]
print(feature_name_reduced)

print("Shape before tfidf: ", X_tfidf.shape)
print("Shape after tfidf and variance threshold:", X_tfidf_reduced.shape)

In [0]:
n_components = 200
svd = TruncatedSVD(n_components=n_components, random_state=2)
lsa_result = svd.fit_transform(X_tfidf_reduced)
lsa_result_nor = normalize(lsa_result, norm='l2', axis=1)
print(f"LSA result shape for K-Means: {lsa_result_nor.shape}")
print(f"LSA result shape for GMM: {lsa_result.shape}")

explained_variance = svd.explained_variance_ratio_.sum()
print(f"Total explained variance: {explained_variance * 100:.2f}%")

In [0]:
# LSA for GMM optimization
n_components = 30
svd = TruncatedSVD(n_components=n_components, random_state=2)
lsa_result_gmm_full = svd.fit_transform(X_tfidf_reduced)
print(f"LSA result shape for GMM_full: {lsa_result_gmm_full.shape}")

explained_variance = svd.explained_variance_ratio_.sum()
print(f"Total explained variance: {explained_variance * 100:.2f}%")

In [0]:
# Save LSA features for visualization (numpy array)
dbutils.fs.mkdirs("/Volumes/scientific_trend_analysis/core/model_artifacts/")
with open("/Volumes/scientific_trend_analysis/core/model_artifacts/lsa_result_nor.pkl", "wb") as f:
    pickle.dump(lsa_result_nor, f)
dbutils.fs.mkdirs("/Volumes/scientific_trend_analysis/core/model_artifacts/")
with open("/Volumes/scientific_trend_analysis/core/model_artifacts/lsa_result.pkl", "wb") as f:
    pickle.dump(lsa_result, f)
dbutils.fs.mkdirs("/Volumes/scientific_trend_analysis/core/model_artifacts/")
with open("/Volumes/scientific_trend_analysis/core/model_artifacts/lsa_result_gmm_full.pkl", "wb") as f:
    pickle.dump(lsa_result_gmm_full, f)

#### K-Means

In [0]:
inertia = []
silhouette_scores = {}

k_range = range(7, 14)
for i in k_range:
    km = KMeans(n_clusters=i, 
                init='k-means++',
                n_init='auto',
                random_state=2,
                max_iter=300)
    km.fit(lsa_result_nor)
    inertia.append(km.inertia_)
    cluster_labels = km.predict(lsa_result_nor)
    
    score = silhouette_score(lsa_result_nor, cluster_labels)
    silhouette_scores[i] = score
    print(f"k={i}, Silhouette Score: {score:.4f}")

# plt.plot(k_range, inertia, marker='o')
# plt.xlabel('Number of cluster', fontsize=14)
# plt.ylabel('Inertia', fontsize=14)
# plt.xticks(list(k_range))

# find the best k
best_k = max(silhouette_scores, key=silhouette_scores.get)
print(f"\nBest k values according to Silhouette Score: {best_k}")


In [0]:
fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot inertia on the first y-axis
color = 'tab:blue'
ax1.set_xlabel('Number of Clusters (k)', fontsize=12)
ax1.set_ylabel('Inertia', fontsize=12, color=color)
ax1.plot(k_range, inertia, marker='o', color=color, linewidth=2, markersize=8, label='Inertia')
ax1.tick_params(axis='y', labelcolor=color)
ax1.grid(True, alpha=0.3)
ax1.set_xticks(list(k_range))

# Create second y-axis for silhouette score
ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Silhouette Score', fontsize=12, color=color)
silhouette_values = [silhouette_scores[k] for k in k_range]
ax2.plot(k_range, silhouette_values, marker='s', color=color, linewidth=2, markersize=8, label='Silhouette Score')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('K-Means: Elbow Plot and Silhouette Score', fontsize=14, fontweight='bold')

# Combine legends
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='best', fontsize=10)

plt.tight_layout()
plt.show()

In [0]:
# final km model
best_k = 12
final_kmeans = KMeans(n_clusters=best_k, random_state=2, init='k-means++', n_init='auto', max_iter=300)
cluster_labels_km = final_kmeans.fit_predict(lsa_result_nor)

# Assign labels to the original dataframe
df_label['Cluster_ID_km'] = cluster_labels_km

silhouette_plot(lsa_result=lsa_result_nor, cluster_labels=cluster_labels_km, best_k=best_k)

In [0]:
print("K-means")
print("-" * 100)
get_top_frequency_words_per_topic(df_label, 'Cluster_ID_km')
get_top_tfidf_words_per_topic(df_label, X_tfidf_reduced, feature_name_reduced, 'Cluster_ID_km')

#### Gaussian Mixture Model - Part 1
Higher Dimensionality + diag covariance

In [0]:
k_range = range(7, 14)

aic_scores, bic_scores = [], []
silhouette_scores = {}
for k in k_range:
    gmm_model = GaussianMixture(
        n_components=k,
        covariance_type='diag',
        random_state=2,
        max_iter=500,
        n_init=10,
        reg_covar=1e-6
    )
    gmm_model.fit(lsa_result)
    aic_scores.append(gmm_model.aic(lsa_result))
    bic_scores.append(gmm_model.bic(lsa_result))

    gmm_cluster_labels = gmm_model.predict(lsa_result)

    score = silhouette_score(lsa_result, gmm_cluster_labels)
    silhouette_scores[k] = score 

    print(f"GMM K={k}, AIC: {gmm_model.aic(lsa_result):.2f}, BIC: {gmm_model.bic(lsa_result):.2f}, Silhouette Score: {score:.4f}")


# find the best k
best_k_ss = max(silhouette_scores, key=silhouette_scores.get)
best_k_bic = k_range[np.argmin(bic_scores)]
print(f"\nBest k values according to Silhouette Score: {best_k_ss}")
print(f"\nBest k values according to BIC: {best_k_bic}")

In [0]:
fig, ax1 = plt.subplots(figsize=(10, 5))

# Plot AIC and BIC scores on the first y-axis
ax1.plot(k_range, aic_scores, marker='o', label='AIC')
ax1.plot(k_range, bic_scores, marker='o', label='BIC')

# Add silhouette score to the plot on a second y-axis
silhouette_score_values = [silhouette_scores[k] for k in k_range]
ax2 = ax1.twinx()
ax2.plot(k_range, silhouette_score_values, marker='o', color='green', label='Silhouette Score')

# best_k_bic = k_range[np.argmin(bic_scores)]
# ax1.axvline(x=best_k_bic, color='red', linestyle='--', label=f'Best K (BIC): {best_k_bic}')

ax1.set_xlabel('Number of cluster (k)', fontsize=14)
ax1.set_ylabel('AIC / BIC', fontsize=14)
ax2.set_ylabel('Silhouette Score', fontsize=14)
ax1.set_title('GMM Model Selection Metrics', fontsize=16)

# Combine legends from both axes
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='best')

plt.show()

In [0]:
# final gmm model
optimal_k = 12 #best_k_bic
final_gmm_model = GaussianMixture(
    n_components=optimal_k,
    covariance_type='diag',
    random_state=2,
    max_iter=500,
    n_init=10,
    reg_covar=1e-6
)
cluster_labels_gmm = final_gmm_model.fit_predict(lsa_result)
probabilities_diag = final_gmm_model.predict_proba(lsa_result)

# Save probabilities for later use
with open("/Volumes/scientific_trend_analysis/core/model_artifacts/gmm_probabilities_diag.pkl", "wb") as f:
    pickle.dump(probabilities_diag, f)

# Assign labels to the original dataframe
df_label['Cluster_ID_gmm_diag'] = cluster_labels_gmm

silhouette_plot(lsa_result=lsa_result, cluster_labels=cluster_labels_gmm, best_k=optimal_k)

In [0]:
plt.figure(figsize=(12, 6))
sns.heatmap(probabilities_diag, cmap='viridis', cbar=True)
plt.xlabel('Cluster')
plt.ylabel('Document')
plt.title('Document-Cluster Probability Heatmap (GMM)')
plt.show()

In [0]:
# For each cluster, show how many papers have different probability levels
n_clusters = probabilities.shape[1]

# Define probability bins
bins = [0, 0.5, 0.6, 0.7, 0.8, 0.9, 0.99, 1.0]
bin_labels = ['<50%', '50-59%', '60-69%', '70-79%', '80-89%', '90-99%', '100%']

# For each cluster, bin the probabilities
cluster_distributions = []
for cluster_id in range(n_clusters):
    cluster_probs = probabilities[:, cluster_id]
    # Bin the probabilities
    binned = pd.cut(cluster_probs, bins=bins, labels=bin_labels, include_lowest=True)
    counts = binned.value_counts().reindex(bin_labels, fill_value=0)
    # Calculate percentages
    percentages = (counts / len(cluster_probs)) * 100
    cluster_distributions.append(percentages)

dist_df = pd.DataFrame(cluster_distributions, index=[f'Cluster {i}' for i in range(n_clusters)])

# Plot stacked bar chart
fig, ax = plt.subplots(figsize=(16, 8))
colors = ['#d73027', '#fc8d59', '#fee090', '#e0f3f8', '#91bfdb', '#4575b4', '#313695']
dist_df.plot(kind='bar', stacked=True, ax=ax, color=colors)
ax.set_xlabel('Cluster', fontsize=18)
ax.set_ylabel('Percentage of Papers (%)', fontsize=18)
ax.set_title('Distribution of Cluster Assignment Probabilities by Cluster (GMM)', fontsize=14)
ax.legend(title='Probability Range', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=16)
plt.xticks(rotation=45, ha='right', fontsize=18)
plt.yticks(fontsize=18)
ax.set_ylim(80, 100)
plt.tight_layout()
plt.show()

# Print statistics for each cluster
print("\n--- Cluster Assignment Certainty Statistics ---")
for cluster_id in range(n_clusters):
    cluster_probs = probabilities[:, cluster_id]
    high_conf = (cluster_probs > 0.9).sum()
    medium_conf = ((cluster_probs > 0.7) & (cluster_probs <= 0.9)).sum()
    low_conf = ((cluster_probs > 0.5) & (cluster_probs <= 0.7)).sum()
    total = len(cluster_probs)
    print(f"Cluster {cluster_id}: >90%: {high_conf} ({high_conf/total*100:.1f}%), 70-90%: {medium_conf} ({medium_conf/total*100:.1f}%), 50-70%: {low_conf} ({low_conf/total*100:.1f}%)")

In [0]:
print("Gaussian Mixture Model")
print("-" * 100)
get_top_frequency_words_per_topic(df_label, 'Cluster_ID_gmm_diag')
get_top_tfidf_words_per_topic(df_label, X_tfidf_reduced, feature_name_reduced, 'Cluster_ID_gmm_diag')

#### Gaussian Mixture Model - Part 2
Lower Dimensionality + full covariance

In [0]:
k_range = range(7, 14)

aic_scores, bic_scores = [], []
silhouette_scores = {}
for k in k_range:
    gmm_model = GaussianMixture(
        n_components=k,
        covariance_type='full',
        random_state=2,
        max_iter=500,
        n_init=10,
        reg_covar=1e-6
    )
    gmm_model.fit(lsa_result_gmm_full)
    aic_scores.append(gmm_model.aic(lsa_result_gmm_full))
    bic_scores.append(gmm_model.bic(lsa_result_gmm_full))

    gmm_cluster_labels = gmm_model.predict(lsa_result_gmm_full)

    score = silhouette_score(lsa_result_gmm_full, gmm_cluster_labels)
    silhouette_scores[k] = score 

    print(f"GMM K={k}, AIC: {gmm_model.aic(lsa_result_gmm_full):.2f}, BIC: {gmm_model.bic(lsa_result_gmm_full):.2f}, Silhouette Score: {score:.4f}")


# find the best k
best_k_ss = max(silhouette_scores, key=silhouette_scores.get)
best_k_bic = k_range[np.argmin(bic_scores)]
print(f"\nBest k values according to Silhouette Score: {best_k_ss}")
print(f"\nBest k values according to BIC: {best_k_bic}")

In [0]:
fig, ax1 = plt.subplots(figsize=(10, 5))

# Plot AIC and BIC scores on the first y-axis
ax1.plot(k_range, aic_scores, marker='o', label='AIC')
ax1.plot(k_range, bic_scores, marker='o', label='BIC')

# Add silhouette score to the plot on a second y-axis
silhouette_score_values = [silhouette_scores[k] for k in k_range]
ax2 = ax1.twinx()
ax2.plot(k_range, silhouette_score_values, marker='o', color='green', label='Silhouette Score')

# best_k_bic = k_range[np.argmin(bic_scores)]
# ax1.axvline(x=best_k_bic, color='red', linestyle='--', label=f'Best K (BIC): {best_k_bic}')

ax1.set_xlabel('Number of cluster (k)', fontsize=14)
ax1.set_ylabel('AIC / BIC', fontsize=14)
ax2.set_ylabel('Silhouette Score', fontsize=14)
ax1.set_title('GMM Model Selection Metrics', fontsize=16)

# Combine legends from both axes
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='best')

plt.show()

In [0]:
# final gmm model
optimal_k = 12 #best_k_bic
final_gmm_model = GaussianMixture(
    n_components=optimal_k,
    covariance_type='full',
    random_state=2,
    max_iter=500,
    n_init=10,
    reg_covar=1e-6
)
cluster_labels_gmm = final_gmm_model.fit_predict(lsa_result_gmm_full)
probabilities_full = final_gmm_model.predict_proba(lsa_result_gmm_full)

# Save probabilities for later use
with open("/Volumes/scientific_trend_analysis/core/model_artifacts/gmm_probabilities_full.pkl", "wb") as f:
    pickle.dump(probabilities_full, f)

# Assign labels to the original dataframe
df_label['Cluster_ID_gmm_full'] = cluster_labels_gmm

silhouette_plot(lsa_result=lsa_result_gmm_full, cluster_labels=cluster_labels_gmm, best_k=optimal_k)

In [0]:
plt.figure(figsize=(12, 6))
sns.heatmap(probabilities_full, cmap='viridis', cbar=True)
plt.xlabel('Cluster')
plt.ylabel('Document')
plt.title('Document-Cluster Probability Heatmap (GMM)')
plt.show()

In [0]:
# For each cluster, show how many papers have different probability levels
n_clusters = probabilities.shape[1]

# Define probability bins
bins = [0, 0.5, 0.6, 0.7, 0.8, 0.9, 0.99, 1.0]
bin_labels = ['<50%', '50-59%', '60-69%', '70-79%', '80-89%', '90-99%', '100%']

# For each cluster, bin the probabilities
cluster_distributions = []
for cluster_id in range(n_clusters):
    cluster_probs = probabilities[:, cluster_id]
    # Bin the probabilities
    binned = pd.cut(cluster_probs, bins=bins, labels=bin_labels, include_lowest=True)
    counts = binned.value_counts().reindex(bin_labels, fill_value=0)
    # Calculate percentages
    percentages = (counts / len(cluster_probs)) * 100
    cluster_distributions.append(percentages)

dist_df = pd.DataFrame(cluster_distributions, index=[f'Cluster {i}' for i in range(n_clusters)])

# Plot stacked bar chart
fig, ax = plt.subplots(figsize=(16, 8))
colors = ['#d73027', '#fc8d59', '#fee090', '#e0f3f8', '#91bfdb', '#4575b4', '#313695']
dist_df.plot(kind='bar', stacked=True, ax=ax, color=colors)
ax.set_xlabel('Cluster', fontsize=18)
ax.set_ylabel('Percentage of Papers (%)', fontsize=18)
ax.set_title('Distribution of Cluster Assignment Probabilities by Cluster (GMM)', fontsize=14)
ax.legend(title='Probability Range', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=16)
plt.xticks(rotation=45, ha='right', fontsize=18)
plt.yticks(fontsize=18)
ax.set_ylim(80, 100)
plt.tight_layout()
plt.show()

# Print statistics for each cluster
print("\n--- Cluster Assignment Certainty Statistics ---")
for cluster_id in range(n_clusters):
    cluster_probs = probabilities[:, cluster_id]
    high_conf = (cluster_probs > 0.9).sum()
    medium_conf = ((cluster_probs > 0.7) & (cluster_probs <= 0.9)).sum()
    low_conf = ((cluster_probs > 0.5) & (cluster_probs <= 0.7)).sum()
    total = len(cluster_probs)
    print(f"Cluster {cluster_id}: >90%: {high_conf} ({high_conf/total*100:.1f}%), 70-90%: {medium_conf} ({medium_conf/total*100:.1f}%), 50-70%: {low_conf} ({low_conf/total*100:.1f}%)")

In [0]:
print("Gaussian Mixture Model")
print("-" * 100)
get_top_frequency_words_per_topic(df_label, 'Cluster_ID_gmm_full')
get_top_tfidf_words_per_topic(df_label, X_tfidf_reduced, feature_name_reduced, 'Cluster_ID_gmm_full')

In [0]:
display(df_label)

In [0]:
spark.createDataFrame(df_label).write.mode("overwrite").option("overwriteSchema", "true") \
                                    .saveAsTable("scientific_trend_analysis.rep.arxiv_clustered")
print(f"Saved {len(df_label)} records with cluster assignments")